# M2 (V5) Controlled Two-Student Study — Colab T4

Runs the approved **M2 / V5 Offline Two-Student Study** under strict split and calibration discipline:
- **Student 1:** `BAAI/bge-reranker-base` (278M XLM-RoBERTa cross-encoder, full FP16-AMP FT)
- **Student 2:** `Qwen/Qwen3.5-0.8B-Base` (800M causal LM, native FP32 LoRA over FP16 base, option-letter logits A–E)
- **Frozen Reference:** `jaredpalmer/kev-0.8b` (off-the-shelf pointer scorer)
- **Splits:** `deepseek_v41_flash_verified_train.jsonl` (`833/840` verified teacher hard labels) · `dev_180.jsonl` (epoch & deployment-seed selection) · `calibrate_180.jsonl` (post-selection temperature scaling) · `holdout_600.jsonl` (`600` unique `source_id`s, evaluated **once** after freezing).

### Setup
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Upload `m2_v5_colab_bundle.zip` to Google Drive root (`MyDrive/`) or upload when prompted in Cell 2.
3. Run Cells 1–4 first (training & calibration). Once satisfied with dev/calibration outputs, run Cell 5 **once** for the frozen `holdout_600` evaluation. All checkpoints and JSON reports save automatically to `/content/drive/MyDrive/self_jev_m2_v5/`.

In [ ]:
# Cell 1: GPU check, Google Drive mount, and clean dependencies
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
!pip install -q --upgrade transformers accelerate safetensors peft

In [ ]:
# Cell 2: Locate and unpack m2_v5_colab_bundle.zip
import os, zipfile
from pathlib import Path

os.environ['HF_HUB_DISABLE_XET'] = '1'
WORK_DIR = Path('/content/m2_v5_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

candidates = [
    Path('/content/m2_v5_colab_bundle.zip'),
    Path('/content/drive/MyDrive/m2_v5_colab_bundle.zip'),
]
bundle = next((p for p in candidates if p.is_file()), None)
if bundle is None:
    from google.colab import files
    print('Upload m2_v5_colab_bundle.zip:')
    uploaded = files.upload()
    bundle = Path('/content') / next(iter(uploaded.keys()))

with zipfile.ZipFile(bundle, 'r') as zf:
    zf.extractall(WORK_DIR)

print('Unpacked bundle from:', bundle)
!ls -lh /content/m2_v5_work/data/teacher/ /content/m2_v5_work/data/holdout/

In [ ]:
# Cell 3: Fast 12-item smoke test (verifies both models, letter IDs, and T4 VRAM)
!python3 /content/m2_v5_work/scripts/train_m2_v5_students.py \
  --data-dir /content/m2_v5_work/data/teacher \
  --train-file deepseek_v41_flash_verified_train.jsonl \
  --out-dir /content/m2_v5_work/smoke_out \
  --student both \
  --smoke-test

In [ ]:
# Cell 4: Full 3-seed training + dev selection + calibrate_180 temperature scaling
# Holdout_600 is NOT loaded or touched in this step.
OUT_DIR = '/content/drive/MyDrive/self_jev_m2_v5'
!python3 /content/m2_v5_work/scripts/train_m2_v5_students.py \
  --data-dir /content/m2_v5_work/data/teacher \
  --train-file deepseek_v41_flash_verified_train.jsonl \
  --out-dir {OUT_DIR} \
  --student both \
  --seeds 42 1024 2026 \
  --bge-epochs 4 --bge-lr 2e-5 \
  --qwen-epochs 4 --qwen-lr 2e-4 --lora-r 16 \
  --grad-accum 8

In [ ]:
# Cell 5: FROZEN SINGLE-PASS EVALUATION on untouched holdout_600.jsonl
# Run ONCE after Cell 4 completes and all checkpoints/temperatures are frozen.
OUT_DIR = '/content/drive/MyDrive/self_jev_m2_v5'
!python3 /content/m2_v5_work/scripts/eval_m2_v5_holdout.py \
  --train-out-dir {OUT_DIR} \
  --holdout-file /content/m2_v5_work/data/holdout/holdout_600.jsonl \
  --data-dir /content/m2_v5_work/data/teacher \
  --kev-vendor-dir /content/m2_v5_work/decision_eval/vendor/kev

In [ ]:
# Cell 6: Print complete side-by-side summary (Dev + Frozen Holdout-600 + McNemar)
import json
from pathlib import Path

OUT_DIR = Path('/content/drive/MyDrive/self_jev_m2_v5')
tr = json.loads((OUT_DIR / 'm2_v5_train_summary.json').read_text())
ho = json.loads((OUT_DIR / 'm2_v5_holdout_report.json').read_text())

print('=== DEV-180 SUMMARY (3 SEEDS) ===')
for name in ['bge', 'qwen']:
    d = tr[name]
    fz = d['frozen_baseline']['dev_calibrated']
    ag = d['aggregate_dev']
    print(f"{name.upper()} ({d['model_id']}, deploy_seed={d['deployment_seed_selected_on_dev']}):")
    print(f"  Zero-shot dev: acc={fz['acc_orig']:.4f} swap={fz['acc_swapped']:.4f} flip={fz['order_flip_rate']:.4f} ece={fz['ece']:.4f} aurc={fz['aurc']:.4f}")
    print(f"  Trained   dev: acc={ag['acc_orig']['mean']:.4f}+/-{ag['acc_orig']['sd']:.4f} flip={ag['order_flip_rate']['mean']:.4f} ece={ag['ece']['mean']:.4f} aurc={ag['aurc']['mean']:.4f}")

print('\n=== FROZEN HOLDOUT-600 RESULTS (SINGLE PASS) ===')
m = ho['models']
for k in ['bge_zeroshot', 'bge_trained_deploy', 'qwen_zeroshot', 'qwen_trained_deploy', 'kev_0.8b_frozen']:
    if k not in m:
        continue
    r = m[k]
    print(f"  {k:22s} | acc={r['acc_orig']:.4f} | swap={r['acc_swapped']:.4f} | flip={r['order_flip_rate']:.4f} | ece={r['ece']:.4f} | aurc={r['aurc']:.4f} | abstain={r['abstain_acc']:.4f} | p50={r['latency_p50_ms']:.1f}ms")

for k in ['bge_trained_3seed_aggregate', 'qwen_trained_3seed_aggregate']:
    if k in m:
        ag = m[k]
        print(f"  {k:28s} | acc={ag['acc_orig']['mean']:.4f}+/-{ag['acc_orig']['sd']:.4f} | flip={ag['order_flip_rate']['mean']:.4f} | ece={ag['ece']['mean']:.4f} | aurc={ag['aurc']['mean']:.4f}")

print('\n=== EXACT MCNEMAR PAIRWISE (HOLDOUT-600) ===')
for pair, res in ho.get('mcnemar_pairwise', {}).items():
    print(f"  {pair}: A_only={res['a_only_correct']}, B_only={res['b_only_correct']}, p={res['exact_p_value']}")
